# Clean FineWeb in 60 seconds

This notebook runs [FineWeb](https://huggingface.co/datasets/HuggingFaceFW/fineweb)'s cleaning recipe on a sample of FineWeb with [pyroid](https://github.com/ao/pyroid), straight from Hugging Face `datasets`:

1. **ftfy** fixes mojibake and other Unicode damage,
2. **fastText language ID** (`lid.176.bin`) keeps English with probability > 0.65,
3. the **Gopher repetition, Gopher quality, C4 and FineWeb quality filters** drop low-quality pages,
4. **MinHash** drops near-duplicates (5-grams, 14 buckets of 8 hashes).

These are the stages, settings and order of datatrove's FineWeb pipeline, and on the same documents the output is the same, document for document (pyroid's test suite checks it on a 10,000-document FineWeb slice). The work runs in Rust on every CPU, on Arrow data: a free CPU runtime is enough.

In [ ]:
# pyroid 1.0 is not on PyPI yet: until it is, replace "pyroid[hf]" with the path or URL of a
# pyroid 1.0 wheel. Runs that set PYROID_NOTEBOOK_DATA (local files, no network) skip this.
import os

if not os.environ.get("PYROID_NOTEBOOK_DATA"):
    %pip install -q "pyroid[hf]" datasets

## Settings

By default the notebook streams the first `N_DOCS` documents of FineWeb's `sample-10BT` config and downloads the fastText model once. To run it on local files, without network (as pyroid's test suite does), set these environment variables before starting Jupyter:

| Variable | Meaning |
|---|---|
| `PYROID_NOTEBOOK_DATA` | a FineWeb parquet file to read instead of streaming |
| `PYROID_LID_MODEL` | the path of `lid.176.bin` |
| `PYROID_NOTEBOOK_DOCS` | how many documents (default 10,000) |
| `PYROID_NOTEBOOK_OUT` | where to write the cleaned parquet file |

In [ ]:
N_DOCS = int(os.environ.get("PYROID_NOTEBOOK_DOCS", "10000"))
LOCAL_DATA = os.environ.get("PYROID_NOTEBOOK_DATA")  # None: stream from the Hub
LID_MODEL = os.environ.get("PYROID_LID_MODEL")  # None: download lid.176.bin once
OUT = os.environ.get("PYROID_NOTEBOOK_OUT", "fineweb-clean.parquet")

## 1. Load a FineWeb sample

Streaming reads only the first documents, not the whole config.

In [ ]:
import time

import datasets
import pyarrow.compute as pc

t0 = time.perf_counter()
if LOCAL_DATA:
    ds = datasets.Dataset.from_parquet(LOCAL_DATA)
    ds = ds.select(range(min(N_DOCS, len(ds))))
else:
    stream = datasets.load_dataset(
        "HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True
    )
    ds = datasets.Dataset.from_list(list(stream.take(N_DOCS)))
text_mb = pc.sum(pc.binary_length(ds.with_format("arrow")["text"])).as_py() / 1e6
print(f"{len(ds):,} documents, {text_mb:.1f} MB of text ({time.perf_counter() - t0:.1f} s)")  # noqa: T201
ds

The language ID model: `lid_model_path` returns `PYROID_LID_MODEL` if set, else downloads fastText's `lid.176.bin` (CC BY-SA 3.0) once into a cache and checks its sha256.

In [ ]:
from pyroid.integrations.recipes import lid_model_path

t0 = time.perf_counter()
lid_model = lid_model_path(LID_MODEL)
print(f"{lid_model} ({time.perf_counter() - t0:.1f} s)")  # noqa: T201

## 2. Clean it

`hf.fineweb` runs ftfy, language ID and the filters as a `Dataset.map` over Arrow batches (pyroid reads the text in place: no Python string is built per document), then MinHash over the survivors, in order. It returns:

- `res.dataset`: the documents kept, with their cleaned text and `language`, `language_score`;
- `res.annotated`: every document, with `keep` and the `reason` it was dropped;
- `res.stats`: counts per stage and per reason, and timings.

In [ ]:
from pyroid.integrations import hf

t0 = time.perf_counter()
res = hf.fineweb(ds, lid_model=lid_model)
seconds = time.perf_counter() - t0
print(res.stats)  # noqa: T201
print(f"\n{len(ds) / seconds:,.0f} documents/s on {os.cpu_count()} CPUs ({seconds:.1f} s)")  # noqa: T201

## 3. What was dropped, and why

Reasons are datatrove's, prefixed with the stage that dropped the document.

In [ ]:
for reason, n in list(res.stats.reasons.items())[:10]:
    print(f"{n:>7,}  {reason}")  # noqa: T201

In [ ]:
import textwrap

ann = res.annotated.with_format("arrow")[:]  # every document, as one Arrow table
dropped = ann.filter(pc.is_valid(ann["reason"]))
for stage, n in res.stats.dropped.items():
    if not n:
        continue
    print(f"--- {stage}: {n:,} dropped, for example")  # noqa: T201
    examples = dropped.filter(pc.starts_with(dropped["reason"], stage)).slice(0, 2)
    for r in examples.select(["reason", "language", "language_score", "text"]).to_pylist():
        lang = f"{r['language']} {r['language_score']:.2f}" if r["language"] else "-"
        print(f"[{r['reason']}; {lang}] {textwrap.shorten(r['text'] or '', 240)}\n")  # noqa: T201

## 4. Save the cleaned documents

In [ ]:
t0 = time.perf_counter()
res.dataset.to_parquet(OUT)
print(f"wrote {len(res.dataset):,} documents to {OUT} ({time.perf_counter() - t0:.1f} s)")  # noqa: T201

## Going further

- Share it: `res.dataset.push_to_hub("<your-name>/fineweb-clean")` (after `huggingface_hub.login()`).
- Use the stages on their own, in your own `map`/`filter` chain:

  ```python
  ds = ds.with_format("arrow")  # Arrow batches: zero-copy
  ds = ds.map(hf.fix_text(), batched=True)  # ftfy
  ds = ds.map(hf.langid(lid_model), batched=True)  # + language, language_score
  ds = ds.filter(hf.quality_mask(), batched=True)  # Gopher, C4 and FineWeb filters
  ```

- Bigger corpora: `hf.fineweb` accepts `batch_size`, `threads` and `num_proc`. MinHash always runs once over the whole dataset, in order, so the result does not depend on them. pyroid already uses every CPU inside each batch, so `num_proc` rarely helps.